# មេរៀន 11 - ពិធីសាស្ត្រ ពីភ្នាក់ងារទៅភ្នាក់ងារ (A2A)


## ការតំឡើង


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## តើពិធីសាស្រ្ត A2A គឺអ្វី?

The **Agent-to-Agent (A2A) protocol** គឺជា​ស្តង់ដា​បើក​មួយ​ដែល​អនុញ្ញាតឱ្យភ្នាក់ងារ AI ទំនាក់ទំនង,
រកឃើញគ្នា និងសហការគ្នា — ទោះបីពួកវាត្រូវបានសាងសង់លើវេទិកាផ្សេងៗ ឬបានផ្ទុក
ដោយសេវាកម្មផ្សេងៗ។

Key concepts:

- **Discovery** – ភ្នាក់ងារបោះពុម្ព *Agent Card* ដែលពិពណ៌នាអំពីសម្ងាត់ភាពរបស់ពួកគេ, making it
  ងាយស្រួលសម្រាប់ភ្នាក់ងារផ្សេងទៀត (ឬអ្នករៀបចំ) ក្នុងការស្វែងរកអ្នកជំនាញដែលត្រឹមត្រូវសម្រាប់ភារកិច្ឆិ។

- **Message Passing** – ភ្នាក់ងារប្តូរសារដែលមានរចនាសម្ព័ន្ធ តាមរយៈពិធីសាស្រ្តរួម, ដូច្នេះ a
  សំណើពីភ្នាក់ងារមួយអាចត្រូវបានយល់ និងបំពេញដោយភ្នាក់ងារមួយផ្សេងទៀត ទោះបីការអនុវត្តខាងក្នុង
  ខុសគ្នាក៏ដោយ។

- **Task Lifecycle** – A2A កំណត់ស្ថានភាពដូចជា *submitted*, *working*, *completed*, និង
  *failed*, ដែលផ្តល់ទិដ្ឋភាពពេញលេញដល់អ្នករៀបចំអំពីរបៀបដែលការងារដែលបានចាត់ចែងកំពុងរីកចម្រើន។

នៅក្នុងមេរៀននេះ យើងសម្របសម្រួលសហការរចនាបែប A2A ដោយភ្ជាប់ភ្នាក់ងារធ្វើដំណើរពិសេសបីរូប
ទៅក្នុងដំណើរការអំពើមួយ ដែលរាល់ភ្នាក់ងារផ្តល់ជំនាញរបស់ខ្លួន និងផ្ញើលទ្ឋផលទៅឱ្យភ្នាក់ងារបន្ទាប់។


## ការបង្កើតភ្នាក់ងារធ្វើដំណើរដែលមានឯកទេស


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ការសហការរវាងភ្នាក់ងារច្រើន តាមលំនាំការងារ

យើងភ្ជាប់ភ្នាក់ងារទាំងបីក្នុងលំនាំការងាររៀងទៅមុខ ដែលស្រដៀងនឹងការផ្ទេរសារ A2A:

1. **CurrencyExchangeAgent** ទទួលសំណើររបស់អ្នកប្រើ ហើយផ្តល់ការណែនាំអំពីរូបិយប័ណ្ណ។
2. **ActivityPlannerAgent** ទទួលបរិបទដែលបានបន្ថែមព័ត៌មាន និងបន្ថែមអនុសាសន៍សកម្មភាព۔
3. **TravelManagerAgent** សម្រួលទិន្នន័យពីទាំងពីរ ទៅជា​សេចក្តីសង្ខេបការធ្វើដំណើរចុងក្រោយ។


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ការយល់ដឹងអំពី A2A ក្នុងបរិយាកាសផលិត

នៅក្នុងបរិយាកាសផលិត វិធានការ A2A បើកឱកាសសម្រាប់ស្ថានភាពឆ្លងសេវាដែលមានសមត្ថភាពខ្លាំង:

| សមត្ថភាព | ការពិពណ៌នា |
|---|---|
| **អន្តរកម្មឆ្លងក្របខ័ណ្ឌ** | ភ្នាក់ងារមួយដែលបានសង់ឡើងដោយក្របខ័ណ្ឌមួយ អាចចាត់តាំងភារកិច្ចទៅភ្នាក់ងារដែលបានសង់ឡើងជាមួយក្របខណ្ឌណាមួយដែលគាំទ្រ A2A, អនុញ្ញាតឱ្យមានការអន្តរកម្មពិតប្រាកដឆ្លងអង្គការ។ |
| **ព្រំកំណត់សេវា** | ភ្នាក់ងារអាចមាននៅក្នុង microservices ផ្ទាល់ខ្លួន, តំបន់ cloud ផ្សេងៗ, ឬក៏អង្គការផ្សេងគ្នា ខណៈពេលដែលនៅតែសហការគ្នាដោយរលូន។ |
| **ការរកឃើញឌីណាមិច** | អ្នករៀបចំអាចស្វែងរកនៅក្នុងបញ្ជី Agent Card នៅពេលរត់ដើម្បីរកឃើញអ្នកជំនាញដែលសមស្របបំផុតសម្រាប់ភារកិច្ចរងបាន។ |
| **ការស្ទ្រីម និង push notifications** | A2A គាំទ្រ Server-Sent Events (SSE) សម្រាប់ការអាប់ដេតដំណើរការពេលពិត និងការជូនដំណឹងបែប push សម្រាប់ភារកិច្ចដែលរយៈពេលវែង។ |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## សង្ខេប

ក្នុងមេរៀននេះ អ្នកបានរៀន:

1. **អ្វីទៅជា A2A protocol** — ស្តង់ដារបើកសម្រាប់ការស្វែងរករវាងភ្នាក់ងារ, ការផ្ញើសារ,
   និងការគ្រប់គ្រងភារកិច្ច.
2. **របៀបបង្កើតភ្នាក់ងារពិសេស** — a Currency Exchange agent, an Activity Planner agent, and a Travel Manager orchestrator.
3. **របៀបភ្ជាប់ភ្នាក់ងារចូលទៅក្នុងដំណើរការ** — ដោយប្រើ `WorkflowBuilder` ដើម្បីម៉ូដែលជាលំដាប់
   ការផ្ទេរសាររវាងភ្នាក់ងារ.
4. **របៀបដែល A2A ធ្វើការ​ក្នុងការផលិត** — អនុញ្ញាតឱ្យមានការសហការអន្តរក្របខណ្ឌ និងអន្តរសេវា
   ជាមួយការស្វែងរកឌីណាមិច និងការអាប់ដេតតាមស្ទ្រីម.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះបីយើងខិតខំប្រឹងប្រែងក្នុងការរកភាពត្រឹមត្រូវក្តី សូមយល់ដឹងថាការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬ មានភាពមិនត្រឹមត្រូវ។ ឯកសារដើមនៅក្នុងភាសាដើមគួរត្រូវបានចាត់ទុកថាជាប្រភពសម្រាប់យោង។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងសូមណែនាំឱ្យប្រើការបកប្រែដោយមនុស្សជំនាញ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬ ការបកប្រែខុសណាមួយ ដែលកើតមានពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
